In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q bertopic sentence-transformers pandas pyarrow

In [ ]:
#!/usr/bin/env python3
"""
bertopic_multilingual.py — Google Colab GPU
===========================================

Multilingual BERTopic analysis:

    multilingual embeddings
        -> UMAP
        -> HDBSCAN
        -> optional outlier reassignment
        -> c-TF-IDF topic labels

Main outputs
------------
1. abstract_embeddings.npy
2. abstract_ids.npy
3. embedding_metadata.json
4. bertopic_topics.parquet
5. topic_info.parquet
6. citation_topic_crosstab.parquet
7. with_topics.parquet

Important
---------
- Embeddings are cached and reused.
- BERTopic clustering runs only once.
- topic_prob is the original HDBSCAN membership strength.
- topic_similarity is cosine similarity to the final topic centroid.
- Reassigned outliers are explicitly identified.
"""

import hashlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd


# ========================================================================== #
# CONFIGURATION
# ========================================================================== #

IN_PARQUET = (
    "/content/drive/MyDrive/COGNOSX/ticks/"
    "preprocessing_data/with_clusters_gold.parquet"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/COGNOSX/ticks/"
    "preprocessing_data/bertopic_gold"
)

OUT_TOPICS = OUTPUT_DIR / "bertopic_topics.parquet"
OUT_INFO = OUTPUT_DIR / "topic_info.parquet"
OUT_CROSSTAB = OUTPUT_DIR / "citation_topic_crosstab.parquet"

OUT_MERGED = (
    "/content/drive/MyDrive/COGNOSX/ticks/"
    "preprocessing_data/with_topics_gold.parquet"
)

EMB_CACHE = OUTPUT_DIR / "abstract_embeddings.npy"
IDS_CACHE = OUTPUT_DIR / "abstract_ids.npy"
META_CACHE = OUTPUT_DIR / "embedding_metadata.json"

# Multilingual model mapping many languages into one embedding space
EMB_MODEL = (
    "sentence-transformers/"
    "paraphrase-multilingual-mpnet-base-v2"
)

USE_GPU_CUML = True

MIN_TOPIC_SIZE = 60
MIN_SAMPLES = 5

UMAP_NEIGHBORS = 15
UMAP_COMPONENTS = 5
UMAP_MIN_DIST = 0.0

RANDOM_STATE = 42
EMBEDDING_BATCH_SIZE = 64

# Reduce HDBSCAN outliers after clustering.
REDUCE_OUTLIERS = True

# 0.0 assigns every outlier to its nearest topic.
# Increase, for example to 0.30, to preserve uncertain outliers.
OUTLIER_THRESHOLD = 0.0

# c-TF-IDF vocabulary configuration
NGRAM_RANGE = (1, 2)
MAX_FEATURES = 200_000

# Abstract must contain more than this number of characters
MIN_ABSTRACT_LENGTH = 1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
Path(OUT_MERGED).parent.mkdir(parents=True, exist_ok=True)


# ========================================================================== #
# OPTIONAL GOOGLE DRIVE AND HUGGING FACE SETUP
# ========================================================================== #

def setup_colab():
    """Mount Google Drive and read the optional Hugging Face token."""

    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except ImportError:
        pass
    except Exception as exc:
        print(
            f"[setup] Google Drive mount skipped: {exc}",
            file=sys.stderr,
        )

    # Keep an existing environment token if one is already available.
    if os.environ.get("HF_TOKEN"):
        return

    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")

        if token:
            os.environ["HF_TOKEN"] = token
            print(
                "[setup] Hugging Face token loaded",
                file=sys.stderr,
            )
    except Exception:
        print(
            "[setup] Hugging Face token not found; "
            "public model download will still work",
            file=sys.stderr,
        )


# ========================================================================== #
# TEXT CLEANING
# ========================================================================== #

def safe_string(value):
    """Convert scalar, list or dictionary values safely to text."""

    if value is None or value is pd.NA:
        return ""

    if isinstance(value, dict):
        return "; ".join(
            f"{safe_string(key)}: {safe_string(val)}"
            for key, val in value.items()
        )

    if isinstance(value, (list, tuple, set, np.ndarray)):
        return "; ".join(
            safe_string(item)
            for item in value
            if safe_string(item)
        )

    try:
        missing = pd.isna(value)

        if isinstance(missing, (bool, np.bool_)) and missing:
            return ""
    except Exception:
        pass

    return str(value).strip()


def validate_input_columns(df):
    required = {
        "oa_id",
        "oa_title",
        "oa_abstract",
        "oa_keywords",
    }

    missing = required.difference(df.columns)

    if missing:
        raise ValueError(
            "Missing required input columns: "
            + ", ".join(sorted(missing))
        )


def load_docs(df):
    """
    Construct multilingual documents from title, abstract and keywords.

    Documents without a sufficiently informative abstract are excluded from
    BERTopic, but remain in the final merged corpus with missing topic values.
    """

    validate_input_columns(df)

    title = df["oa_title"].map(safe_string)
    abstract = df["oa_abstract"].map(safe_string)
    keywords = df["oa_keywords"].map(safe_string)

    documents = (
        title.str.rstrip(" .")
        + ". "
        + abstract.str.rstrip(" .")
        # + ". "
        # + keywords.str.rstrip(" .")
    ).str.strip(" .")

    usable = (
        abstract.str.len().gt(MIN_ABSTRACT_LENGTH)
        & documents.str.len().gt(MIN_ABSTRACT_LENGTH)
        & df["oa_id"].notna()
    )

    sub = df.loc[usable, ["oa_id"]].copy()
    sub["_doc"] = documents.loc[usable]

    if sub.empty:
        raise ValueError(
            "No documents have a usable abstract longer than "
            f"{MIN_ABSTRACT_LENGTH} characters."
        )

    duplicated = sub["oa_id"].duplicated(keep=False)

    if duplicated.any():
        examples = (
            sub.loc[duplicated, "oa_id"]
            .astype(str)
            .drop_duplicates()
            .head(10)
            .tolist()
        )

        raise ValueError(
            "oa_id must be unique before BERTopic. "
            f"Example duplicated IDs: {examples}"
        )

    print(
        f"[docs] {len(sub):,} documents with usable abstracts "
        f"out of {len(df):,} rows",
        file=sys.stderr,
    )

    if "oa_language" in df.columns:
        language_mix = (
            df.loc[usable, "oa_language"]
            .fillna("unknown")
            .value_counts()
            .head(10)
        )

        print(
            "[docs] language mix:\n"
            + language_mix.to_string(),
            file=sys.stderr,
        )

    ids = sub["oa_id"].tolist()
    docs = sub["_doc"].tolist()

    return ids, docs


# ========================================================================== #
# EMBEDDING CACHE
# ========================================================================== #

def corpus_fingerprint(ids, docs):
    """
    Create a fingerprint from IDs, texts, model and normalization settings.

    The cache is invalidated if:
    - document order changes;
    - document text changes;
    - IDs change;
    - the embedding model changes.
    """

    hasher = hashlib.sha256()
    hasher.update(EMB_MODEL.encode("utf-8"))
    hasher.update(b"normalize_embeddings=True")

    for identifier, document in zip(ids, docs):
        for value in (safe_string(identifier), document):
            encoded = value.encode("utf-8", errors="replace")
            hasher.update(len(encoded).to_bytes(8, "little"))
            hasher.update(encoded)

    return hasher.hexdigest()


def normalize_rows(matrix):
    """L2-normalize embedding rows for cosine similarity."""

    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0

    return matrix / norms


def cache_is_valid(ids, docs):
    required_files = [
        EMB_CACHE,
        IDS_CACHE,
        META_CACHE,
    ]

    if not all(path.exists() for path in required_files):
        return False

    try:
        with META_CACHE.open("r", encoding="utf-8") as handle:
            metadata = json.load(handle)

        expected_fingerprint = corpus_fingerprint(ids, docs)

        if metadata.get("fingerprint") != expected_fingerprint:
            print(
                "[emb] document or model fingerprint changed",
                file=sys.stderr,
            )
            return False

        cached_ids = np.load(
            IDS_CACHE,
            allow_pickle=True,
        ).tolist()

        if cached_ids != ids:
            print(
                "[emb] cached ID order does not match",
                file=sys.stderr,
            )
            return False

        embeddings = np.load(
            EMB_CACHE,
            mmap_mode="r",
        )

        if embeddings.shape[0] != len(ids):
            print(
                "[emb] cached embedding row count does not match",
                file=sys.stderr,
            )
            return False

        return True

    except Exception as exc:
        print(
            f"[emb] cache validation failed: {exc}",
            file=sys.stderr,
        )
        return False


def get_embeddings(ids, docs):
    """Load valid cached embeddings or compute them once."""

    if cache_is_valid(ids, docs):
        print(
            "[emb] loading cached multilingual embeddings",
            file=sys.stderr,
        )

        embeddings = np.load(EMB_CACHE)
        return normalize_rows(embeddings)

    import torch
    from sentence_transformers import SentenceTransformer

    device = "cuda" if torch.cuda.is_available() else "cpu"

    if device == "cpu":
        print(
            "[emb] CUDA unavailable; embedding on CPU",
            file=sys.stderr,
        )
    else:
        gpu_name = torch.cuda.get_device_name(0)
        print(
            f"[emb] using GPU: {gpu_name}",
            file=sys.stderr,
        )

    print(
        f"[emb] encoding {len(docs):,} documents with {EMB_MODEL}",
        file=sys.stderr,
    )

    model = SentenceTransformer(
        EMB_MODEL,
        device=device,
        token=os.environ.get("HF_TOKEN"),
    )

    embeddings = model.encode(
        docs,
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    embeddings = np.asarray(
        embeddings,
        dtype=np.float32,
    )

    np.save(EMB_CACHE, embeddings)

    np.save(
        IDS_CACHE,
        np.asarray(ids, dtype=object),
        allow_pickle=True,
    )

    metadata = {
        "embedding_model": EMB_MODEL,
        "number_of_documents": len(ids),
        "embedding_dimensions": int(embeddings.shape[1]),
        "normalized": True,
        "fingerprint": corpus_fingerprint(ids, docs),
    }

    with META_CACHE.open("w", encoding="utf-8") as handle:
        json.dump(
            metadata,
            handle,
            indent=2,
            ensure_ascii=False,
        )

    print(
        f"[emb] embeddings cached at {EMB_CACHE}",
        file=sys.stderr,
    )

    return embeddings


# ========================================================================== #
# BERTopic COMPONENTS
# ========================================================================== #

def build_dimensionality_and_cluster_models(number_of_docs):
    """Create GPU models, with a complete CPU fallback."""

    n_neighbors = min(
        UMAP_NEIGHBORS,
        max(2, number_of_docs - 1),
    )

    if USE_GPU_CUML:
        try:
            from cuml.cluster import HDBSCAN as cuHDBSCAN
            from cuml.manifold import UMAP as cuUMAP

            umap_model = cuUMAP(
                n_neighbors=n_neighbors,
                n_components=UMAP_COMPONENTS,
                min_dist=UMAP_MIN_DIST,
                metric="cosine",
                random_state=RANDOM_STATE,
                output_type="numpy",
            )

            hdbscan_model = cuHDBSCAN(
                min_cluster_size=MIN_TOPIC_SIZE,
                min_samples=MIN_SAMPLES,
                metric="euclidean",
                cluster_selection_method="eom",
                prediction_data=True,
                output_type="numpy",
            )

            print(
                "[bertopic] using cuML GPU UMAP and HDBSCAN",
                file=sys.stderr,
            )

            return umap_model, hdbscan_model, "GPU"

        except Exception as exc:
            print(
                f"[bertopic] cuML unavailable: {exc}",
                file=sys.stderr,
            )
            print(
                "[bertopic] switching to CPU UMAP and HDBSCAN",
                file=sys.stderr,
            )

    from hdbscan import HDBSCAN
    from umap import UMAP

    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=UMAP_COMPONENTS,
        min_dist=UMAP_MIN_DIST,
        metric="cosine",
        random_state=RANDOM_STATE,
        low_memory=True,
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=MIN_TOPIC_SIZE,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
        core_dist_n_jobs=-1,
    )

    return umap_model, hdbscan_model, "CPU"


def convert_probabilities(probabilities, number_of_docs):
    """Convert BERTopic/cuML output to a one-dimensional NumPy array."""

    if probabilities is None:
        return np.full(
            number_of_docs,
            np.nan,
            dtype=np.float32,
        )

    # CuPy arrays
    if hasattr(probabilities, "get"):
        probabilities = probabilities.get()

    # cuDF or pandas objects
    elif hasattr(probabilities, "to_numpy"):
        probabilities = probabilities.to_numpy()

    probabilities = np.asarray(probabilities)

    if probabilities.ndim == 2:
        # This should normally not occur because
        # calculate_probabilities=False.
        probabilities = probabilities.max(axis=1)

    probabilities = probabilities.reshape(-1)

    if len(probabilities) != number_of_docs:
        raise ValueError(
            "The probability output does not align with the documents: "
            f"{len(probabilities):,} probabilities for "
            f"{number_of_docs:,} documents."
        )

    return probabilities.astype(np.float32)


def calculate_topic_similarity(
    embeddings,
    original_topics,
    final_topics,
):
    """
    Calculate cosine similarity to final assigned topic centroids.

    Centroids are estimated using original non-outlier HDBSCAN members. This
    prevents reassigned outliers from moving the centroid used to score them.
    """

    embeddings = normalize_rows(embeddings)
    original_topics = np.asarray(original_topics, dtype=int)
    final_topics = np.asarray(final_topics, dtype=int)

    scores = np.full(
        len(final_topics),
        np.nan,
        dtype=np.float32,
    )

    topic_ids = sorted(
        topic
        for topic in set(final_topics.tolist())
        if topic != -1
    )

    for topic_id in topic_ids:
        original_members = original_topics == topic_id

        # Defensive fallback if a topic lacks original members.
        if not original_members.any():
            original_members = final_topics == topic_id

        centroid = embeddings[original_members].mean(axis=0)
        centroid_norm = np.linalg.norm(centroid)

        if centroid_norm == 0:
            continue

        centroid = centroid / centroid_norm

        assigned_members = final_topics == topic_id

        scores[assigned_members] = (
            embeddings[assigned_members] @ centroid
        ).astype(np.float32)

    return scores


# ========================================================================== #
# FIT BERTOPIC ONCE
# ========================================================================== #

def fit_bertopic(docs, embeddings):
    from bertopic import BERTopic
    from bertopic.vectorizers import ClassTfidfTransformer
    from sklearn.feature_extraction.text import CountVectorizer

    umap_model, hdbscan_model, backend = (
        build_dimensionality_and_cluster_models(len(docs))
    )

    vectorizer_model = CountVectorizer(
        ngram_range=NGRAM_RANGE,
        min_df=1,
        max_features=MAX_FEATURES,
        lowercase=True,
        token_pattern=r"(?u)\b[^\W\d_][^\W_]+\b",
    )

    ctfidf_model = ClassTfidfTransformer(
        reduce_frequent_words=True,
    )

    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        min_topic_size=MIN_TOPIC_SIZE,
        calculate_probabilities=False,
        low_memory=True,
        verbose=True,
    )

    print(
        f"[bertopic] fitting once using the {backend} backend",
        file=sys.stderr,
    )

    original_topics, probabilities = topic_model.fit_transform(
        docs,
        embeddings=embeddings,
    )

    original_topics = np.asarray(
        original_topics,
        dtype=int,
    )

    probabilities = convert_probabilities(
        probabilities,
        len(docs),
    )

    original_outlier_rate = np.mean(original_topics == -1)

    print(
        f"[bertopic] initial outlier rate: "
        f"{original_outlier_rate:.1%}",
        file=sys.stderr,
    )

    final_topics = original_topics.copy()

    if REDUCE_OUTLIERS and np.any(original_topics == -1):
        non_outlier_topics = set(original_topics) - {-1}

        if not non_outlier_topics:
            print(
                "[bertopic] all documents are outliers; "
                "outlier reduction cannot be performed",
                file=sys.stderr,
            )
        else:
            print(
                "[bertopic] reducing outliers using "
                "multilingual embedding similarity",
                file=sys.stderr,
            )

            final_topics = np.asarray(
                topic_model.reduce_outliers(
                    docs,
                    original_topics.tolist(),
                    strategy="embeddings",
                    embeddings=embeddings,
                    threshold=OUTLIER_THRESHOLD,
                ),
                dtype=int,
            )

            # Recalculate c-TF-IDF labels from final assignments.
            # This does not repeat UMAP or HDBSCAN.
            topic_model.update_topics(
                docs,
                topics=final_topics.tolist(),
                vectorizer_model=vectorizer_model,
                ctfidf_model=ctfidf_model,
            )

            # Keep the model's document assignments synchronized.
            topic_model.topics_ = final_topics.tolist()

    final_outlier_rate = np.mean(final_topics == -1)

    print(
        f"[bertopic] final outlier rate: "
        f"{final_outlier_rate:.1%}",
        file=sys.stderr,
    )

    similarities = calculate_topic_similarity(
        embeddings=embeddings,
        original_topics=original_topics,
        final_topics=final_topics,
    )

    return {
        "model": topic_model,
        "original_topics": original_topics,
        "final_topics": final_topics,
        "probabilities": probabilities,
        "similarities": similarities,
    }


# ========================================================================== #
# OUTPUT TABLES
# ========================================================================== #

def build_topic_info(model):
    """Create a clean topic-level information table."""

    info = model.get_topic_info().copy()

    if "Representation" in info.columns:
        info["top_terms"] = info["Representation"].apply(
            lambda terms: (
                " | ".join(map(str, terms))
                if isinstance(terms, (list, tuple, np.ndarray))
                else safe_string(terms)
            )
        )

    info = info.rename(
        columns={
            "Topic": "topic_id",
            "Count": "size",
            "Name": "topic_label",
        }
    )

    preferred_columns = [
        "topic_id",
        "topic_label",
        "size",
        "top_terms",
    ]

    remaining_columns = [
        column
        for column in info.columns
        if column not in preferred_columns
    ]

    return info[
        [
            column
            for column in preferred_columns
            if column in info.columns
        ]
        + remaining_columns
    ]


def make_citation_topic_crosstab(out):
    """
    Build a long-form cross-tab between citation clusters and BERTopic topics.
    """

    required = {"cluster_id", "topic_id"}

    if not required.issubset(out.columns):
        print(
            "[reconcile] cluster_id not found; cross-tab skipped",
            file=sys.stderr,
        )
        return None

    both = out.loc[
        out["cluster_id"].notna()
        & out["topic_id"].notna()
        & out["cluster_id"].ne(-1)
        & out["topic_id"].ne(-1),
        ["topic_id", "cluster_id"],
    ].copy()

    if both.empty:
        print(
            "[reconcile] no rows have both valid cluster and topic IDs",
            file=sys.stderr,
        )
        return None

    cross_tab = (
        both.groupby(
            ["topic_id", "cluster_id"],
            dropna=False,
        )
        .size()
        .rename("paper_count")
        .reset_index()
    )

    cross_tab["topic_size_in_crosstab"] = (
        cross_tab.groupby("topic_id")["paper_count"]
        .transform("sum")
    )

    cross_tab["share_within_topic"] = (
        cross_tab["paper_count"]
        / cross_tab["topic_size_in_crosstab"]
    )

    cross_tab = cross_tab.sort_values(
        ["topic_id", "paper_count"],
        ascending=[True, False],
    )

    dominant = (
        cross_tab.groupby("topic_id", as_index=False)
        .first()
    )

    mean_concentration = dominant[
        "share_within_topic"
    ].mean()

    weighted_concentration = np.average(
        dominant["share_within_topic"],
        weights=dominant["topic_size_in_crosstab"],
    )

    from sklearn.metrics import (
        adjusted_rand_score,
        normalized_mutual_info_score,
    )

    topic_labels = both["topic_id"].astype(str)
    cluster_labels = both["cluster_id"].astype(str)

    nmi = normalized_mutual_info_score(
        topic_labels,
        cluster_labels,
    )

    ari = adjusted_rand_score(
        topic_labels,
        cluster_labels,
    )

    print(
        "[reconcile] topic → dominant citation-cluster concentration:",
        file=sys.stderr,
    )
    print(
        f"  unweighted mean: {mean_concentration:.1%}",
        file=sys.stderr,
    )
    print(
        f"  paper-weighted mean: {weighted_concentration:.1%}",
        file=sys.stderr,
    )
    print(
        f"  normalized mutual information: {nmi:.3f}",
        file=sys.stderr,
    )
    print(
        f"  adjusted Rand index: {ari:.3f}",
        file=sys.stderr,
    )

    return cross_tab


# ========================================================================== #
# MAIN
# ========================================================================== #

def main():
    setup_colab()

    input_path = Path(IN_PARQUET)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Input file not found: {input_path}"
        )

    df = pd.read_parquet(input_path)

    print(
        f"[input] {len(df):,} rows loaded",
        file=sys.stderr,
    )

    ids, docs = load_docs(df)
    embeddings = get_embeddings(ids, docs)

    result = fit_bertopic(
        docs=docs,
        embeddings=embeddings,
    )

    model = result["model"]
    original_topics = result["original_topics"]
    final_topics = result["final_topics"]
    probabilities = result["probabilities"]
    similarities = result["similarities"]

    info = build_topic_info(model)

    label_map = dict(
        zip(
            info["topic_id"],
            info["topic_label"],
        )
    )

    topics_df = pd.DataFrame(
        {
            "oa_id": ids,
            "document": docs,
            "topic_id": final_topics,
            "topic_label": pd.Series(final_topics).map(label_map),
            "topic_prob": probabilities,
            "topic_similarity": similarities,
            "topic_was_outlier": original_topics == -1,
            "topic_reassigned": (
                (original_topics == -1)
                & (final_topics != -1)
            ),
        }
    )

    # Nullable integer preserves missing topic IDs after the merge.
    topics_df["topic_id"] = (
        topics_df["topic_id"].astype("Int64")
    )

    topics_df.to_parquet(
        OUT_TOPICS,
        index=False,
    )

    info.to_parquet(
        OUT_INFO,
        index=False,
    )

    print(
        f"[write] topic assignments: {OUT_TOPICS}",
        file=sys.stderr,
    )
    print(
        f"[write] topic information: {OUT_INFO}",
        file=sys.stderr,
    )

    # Remove stale topic columns if the script is rerun.
    generated_columns = [
        "topic_id",
        "topic_label",
        "topic_prob",
        "topic_similarity",
        "topic_was_outlier",
        "topic_reassigned",
    ]

    out = df.drop(
        columns=[
            column
            for column in generated_columns
            if column in df.columns
        ]
    )

    out = out.merge(
        topics_df,
        on="oa_id",
        how="left",
        validate="many_to_one",
    )

    out.to_parquet(
        OUT_MERGED,
        index=False,
    )

    print(
        f"[write] merged corpus: {OUT_MERGED}",
        file=sys.stderr,
    )

    cross_tab = make_citation_topic_crosstab(out)

    if cross_tab is not None:
        cross_tab.to_parquet(
            OUT_CROSSTAB,
            index=False,
        )

        print(
            f"[write] citation/topic cross-tab: {OUT_CROSSTAB}",
            file=sys.stderr,
        )

    number_of_topics = int(
        info.loc[info["topic_id"].ne(-1), "topic_id"].nunique()
    )

    final_outliers = float(
        topics_df["topic_id"].eq(-1).mean()
    )

    print(
        "\n========== COMPLETE ==========",
        file=sys.stderr,
    )
    print(
        f"Topics: {number_of_topics:,}",
        file=sys.stderr,
    )
    print(
        f"Final outliers: {final_outliers:.1%}",
        file=sys.stderr,
    )
    print(
        f"Documents assigned: {len(topics_df):,}",
        file=sys.stderr,
    )


if __name__ == "__main__":
    main()

In [ ]:

# [ ]

# import pandas as pd
# pd.set_option('display.max_columns', None)
# from google.colab import drive
# drive.mount('/content/drive')

# Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

# [ ]

# !pip install -q bertopic sentence-transformers pandas pyarrow

# [ ]

# Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

# [input] 103,737 rows loaded
# [docs] 82,649 documents with usable abstracts out of 103,737 rows
# [docs] language mix:
# oa_language
# en         73703
# unknown     2496
# pt          1939
# es          1536
# ru           763
# tr           405
# fr           358
# de           287
# cs           129
# id           116
# [emb] loading cached multilingual embeddings
# [bertopic] using cuML GPU UMAP and HDBSCAN
# [bertopic] fitting once using the GPU backend
# 2026-08-04 09:47:29,055 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
# 2026-08-04 09:47:33,985 - BERTopic - Dimensionality - Completed ✓
# 2026-08-04 09:47:33,998 - BERTopic - Cluster - Start clustering the reduced embeddings
# 2026-08-04 09:47:35,332 - BERTopic - Cluster - Completed ✓
# 2026-08-04 09:47:35,384 - BERTopic - Representation - Fine-tuning topics using representation models.
# 2026-08-04 09:48:42,749 - BERTopic - Representation - Completed ✓
# [bertopic] initial outlier rate: 39.9%
# [bertopic] reducing outliers using multilingual embedding similarity
# 2026-08-04 09:49:00,059 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
# [bertopic] final outlier rate: 0.0%
# [write] topic assignments: /content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/bertopic2/bertopic_topics.parquet
# [write] topic information: /content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/bertopic2/topic_info.parquet
# [write] merged corpus: /content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_topics.parquet
# [reconcile] topic → dominant citation-cluster concentration:
#   unweighted mean: 70.4%
#   paper-weighted mean: 73.6%
#   normalized mutual information: 0.420
#   adjusted Rand index: 0.098
# [write] citation/topic cross-tab: /content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/bertopic2/citation_topic_crosstab.parquet

# ========== COMPLETE ==========
# Topics: 202
# Final outliers: 0.0%
# Documents assigned: 82,649


In [ ]:
import pandas as pd

# Using the path for the merged output defined previously
file_path = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/bertopic2/topic_info.parquet"

# Load the parquet file into a DataFrame
exploration_df = pd.read_parquet(file_path)

# Display basic info and the first few rows
print("Dataset Information:")
display(exploration_df.info())

print("\nFirst 5 rows:")
display(exploration_df.head(60))


In [ ]:
from google.colab import data_table

data_table.DataTable(
    exploration_df, include_index=False, num_rows_per_page=30
)

In [ ]:
import pandas as pd

# Using the path for the merged output defined previously
file_path = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/bertopic2/bertopic_topics.parquet"

# Load the parquet file into a DataFrame
exploration_df = pd.read_parquet(file_path)

# Display basic info and the first few rows
print("Dataset Information:")
display(exploration_df.info())

print("\nFirst 5 rows:")
display(exploration_df)


In [ ]:
from google.colab import data_table

data_table.DataTable(
    exploration_df, include_index=False, num_rows_per_page=30
)